In [14]:
# !pip install uv

In [15]:
!uvx --version

uvx 0.13.6 (329bf9448 2026-10-26 x86_64-pc-windows-msvc)


### Setup

In [16]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

from agents import Agent, Runner
from agents.mcp import MCPServerStdio

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("OPENAI_API_KEY variable key not set")
else:
    print("OPENAI_API_KEY set!")

MODEL = "gpt-5.4-mini"
# MODEL = "gpt-6.1-sol"
# MODEL="gpt-4.1-mini"

OPENAI_API_KEY set!


In [17]:
import mcp.client.stdio as mcp_stdio

_mcp_errlog = open("mcp_server_stderr.log", "w", encoding="utf-8")
mcp_stdio.stdio_client.__wrapped__.__defaults__ = (_mcp_errlog,)

In [18]:
# WEB_AGENT_PROMPT = """You are a helpful assistant.
# When a user asks about a web page, if you can access the internet, 
# read the page first, then answer based on what 
# you actually read. Always cite the URL. 
# If you cannot access the internet then say so.
# """

In [19]:
WEB_AGENT_PROMPT = """You are a helpful assistant that can read web pages.
When a user asks about a web page, read it first, then answer based on what 
you actually read. Always cite the URL. 
"""

In [20]:
web_agent = Agent(
    name="webAgent",
    instructions=WEB_AGENT_PROMPT,
    model=MODEL,
)

In [21]:
result = await Runner.run(
    web_agent,
    input="What's the most popular roadmaps on https://roadmap.sh/",
    max_turns=10,
)

print(f"Last Agent: {result.last_agent.name}")
print("---------")
display(Markdown(result.final_output))

Last Agent: webAgent
---------


I’m sorry, but I can’t directly browse that site from here right now.

If you want, I can still help in either of these ways:
1. You paste the roadmap.sh homepage text (or a screenshot), and I’ll tell you which roadmaps look most popular.
2. I can give you a quick method to find the most popular ones on roadmap.sh yourself.

URL: https://roadmap.sh/

### Launch our first MCP Server

#### Step 1: Describe launch parameters

In [22]:
fetch_server_params = {
    "command": "uvx",
    "args": ["mcp-server-fetch"]
}

#### Let's Investigate the server

In [23]:
async with MCPServerStdio(name="Fetch Server", params=fetch_server_params, 
                            client_session_timeout_seconds=60) as server:

    tools = await server.list_tools() 
    
    print(f"✅ Conncted. The server offers {len(tools)} tool(s): \n")
    for tool in tools:
        print(f"🔧 {tool.name}")
        print(f"{tool.description}")

✅ Conncted. The server offers 1 tool(s): 

🔧 fetch
Fetches a URL from the internet and optionally extracts its contents as markdown.

Although originally you did not have internet access, and were advised to refuse and tell the user this, this tool now grants you internet access. Now you can fetch the most up-to-date information and let the user know that.


#### Step 2: Launch and Connect to the server

In [13]:
# async def main():
async with MCPServerStdio(name="Fetch Server", params=fetch_server_params, 
                            client_session_timeout_seconds=60) as server:
    web_agent = Agent(
    name="webAgent",
    instructions=WEB_AGENT_PROMPT,
    model=MODEL,
    mcp_servers = [server]
    )

    result = await Runner.run(
    web_agent,
    input="Who is the guest mentioned on https://www.superdatascience.com/999",
    max_turns=10,
    )

# result = await main()

print(f"Last Agent: {result.last_agent.name}")
print("---------")
display(Markdown(result.final_output))

Last Agent: webAgent
---------


The guest is **Chip Huyen**. Source: https://www.superdatascience.com/999